## Setup

In [ ]:
from __future__ import annotations

import os
import time
from pynput import keyboard

from conversation import conversation_with_AI
from core.config import DEFAULT_TTS_LANGUAGE, get_paths
from core.llm import load_model
from core.memory import MesmerlaMemory
from core.tts import load_xtts, speak_as_mesmerla, play_audio

import numpy as np
import soundfile as sf
import torch
import TTS.tts.models.xtts as xtts_mod

In [ ]:
def patched_load_audio(audiopath, sampling_rate):
    audio, sr = sf.read(audiopath, dtype="float32")
    if audio.ndim > 1:
        audio = np.mean(audio, axis=1)  # stereo -> mono

    audio = torch.from_numpy(audio).unsqueeze(0)

    if sr != sampling_rate:
        import torchaudio.functional as F
        audio = F.resample(audio, sr, sampling_rate)

    return audio

xtts_mod.load_audio = patched_load_audio

In [ ]:
# Settings
personality = "Mesmerla"
mode = "reflective"
tts_language = DEFAULT_TTS_LANGUAGE

memory = MesmerlaMemory(personality)
memory.reset()

In [ ]:
_, _, _, _, model_path = get_paths(personality)
llm = load_model(
    model_path,
    n_ctx=2048,
    n_threads=os.cpu_count(),
    n_batch=64,
    verbose=False,
)

In [ ]:
# Preload XTTS once so first reply is not painfully slow.
load_xtts()

In [ ]:
ref_audio_path, ref_text_path, output_path, _, _ = get_paths("Zhongli")

response = speak_as_mesmerla(
        text="Je teste. Est-ce que tu m'entends ?",
        ref_audio_path=ref_audio_path,
        ref_text_path=ref_text_path,
        output_path=output_path,
        language='fr',
    )

if response.get("status") == "ok":
        play_audio(response["output_path"])
else:
    print("⚠️ TTS error:", response)

## Converse

In [ ]:
response = conversation_with_AI(llm, personality="Mesmerla", mode="concise", verbose=True, tts_language="en")

In [ ]:
from pynput import keyboard
import time
# Global control
continue_conversation = True

# Define keypress handling
def on_key_press(key):
    global continue_conversation
    if hasattr(key, 'char') and key.char == 'q':
        continue_conversation = False
        print("🛑 Stopping conversation loop... Pressed 'q'")
        return False  # Stops listener

print("🔁 Press 'q' at any time to stop.")
listener = keyboard.Listener(on_press=on_key_press)
listener.start()

try:
    while continue_conversation:
        conversation_with_AI(llm, personality="Mesmerla", mode="concise", verbose=False, tts_language="en")
        print("⏳ Listening again...")
        time.sleep(1)
except KeyboardInterrupt:
    print("🛑 Stopping conversation loop... (KeyboardInterrupt)")
    continue_conversation = False

listener.join()

## Work testing

In [ ]:
memory = MesmerlaMemory(style="Mesmerla")
memory.load()

In [ ]:
# View entries
for entry in memory.entries:
    print(f"User: {entry['user']}\nMesmerla: {entry['response']}\n")

In [ ]:
memory.reset()

In [ ]:
stop_mesmerla_server()

## Finetuning work

In [ ]:
from pathlib import Path
import json
import textwrap

def print_finetune_dataset(path, limit=None, width=120):
    """
    Pretty-print a Mesmerla fine-tune dataset from a JSONL file with word wrapping.
    
    Parameters:
        path (str): Path to the .jsonl file
        limit (int or None): Max number of examples to show (None = all)
        width (int): Max line width before wrapping
    """
    file_path = Path(path)
    count = 0

    with file_path.open("r", encoding="utf-8") as f:
        for i, line in enumerate(f, 1):
            example = json.loads(line)
            print(f"🔹 Example {i}")
            print(textwrap.fill(example["prompt"], width=width))
            print(f"💬 {textwrap.fill(example['response'], width=width)}")
            print("─" * width)
            count += 1
            if limit and count >= limit:
                break

In [ ]:
print_finetune_dataset("finetuning/mesmerla_finetune_set_batch10.jsonl")

In [ ]:
from pathlib import Path

# Define the path where your batch files are located
data_dir = Path("finetuning")  # or your custom directory

# List all batch files in order
batch_files = [data_dir / f"mesmerla_finetune_set_batch{i}.jsonl" for i in range(1, 11)]

# Output file
output_file = data_dir / "mesmerla_dataset.jsonl"

# Combine them
with output_file.open("w", encoding="utf-8") as outfile:
    for file in batch_files:
        with file.open("r", encoding="utf-8") as infile:
            lines = infile.readlines()
            outfile.writelines(lines)

print(f"✅ Merged {len(batch_files)} batches into {output_file.name}")


## conversing by chat

In [1]:
from core.config import get_paths
from core.llm import load_model
from text_convo import text_conversation
from core.memory import MesmerlaMemory

In [2]:
# Load the model path dynamically
_, _, _, _, model_path = get_paths("HuTao")  # Or "HuTao", "Zhongli"
llm = load_model(model_path, verbose=False)

🧠 Loading model for Mesmerla...


llama_context: n_ctx_seq (2048) < n_ctx_train (8192) -- the full capacity of the model will not be utilized


✅ Model loaded in 16.73s 
loaded C:\Users\aberl\Desktop\Projet Code\Mesmerla_AI\AI-ssistant\models\Meta-Llama-3-8B-Instruct-Q4_K_M.gguf


In [3]:

user_message = """That's great to hear! do you now want to also have your voice be streamed ?"""

# Get reply
reply = text_conversation(llm, user_message, personality="Mesmerla", mode="reflective", verbose=False, jupyter_notebook=True)

#print("\n", prompt)

In [ ]:
memory = MesmerlaMemory(style="Mesmerla")
memory.load()
# View entries
for entry in memory.entries:
    print(f"User: {entry['user']}\nMesmerla: {entry['response']}\n")

In [ ]:
memory.reset()

In [ ]:
import inspect
import llama_cpp

print("llama-cpp-python:", llama_cpp.__version__)
print("llm type:", type(llm))
print("llm module:", type(llm).__module__)
print(inspect.signature(llm.create_chat_completion))

In [ ]:
test_response = llm.create_chat_completion(
    messages=[
        {"role": "user", "content": "Say hello in five words."},
    ],
    max_tokens=20,
    stream=True,
)

print("Response type:", type(test_response))
print("Response:", test_response)

In [ ]:
import core
print(core.llm.__file__)
print(inspect.getsource(core.llm.generate_response_stream))